In [ ]:
# Libraries:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
import joblib

from imblearn.over_sampling import SMOTENC

In [ ]:
# Read in PreGame test data for training
os.chdir("/path/to/PreGame/testing/data/")

X_concat = pd.read_feather("PreGame_testing_matrix.feather").to_numpy()
Y_true = pd.read_csv("PreGame_testing_labels.tsv", sep='\t').values.ravel()
sample_num = X_concat.shape[0]

In [ ]:
# Define variables: 
skf = StratifiedKFold(n_splits=10)
smt = SMOTENC(categorical_features=[2],)
K = 100

models = {}
dataset = {}

# Train model: 
for k in range(K):
    X_resampled, y_resampled = smt.fit_resample(X_concat, Y_true)

    clf = RandomForestClassifier(n_estimators = 200, max_depth=5, class_weight={0:1,1:5}, random_state=42)

    clf.fit(X_resampled, y_resampled)
    models[k] = clf
    dataset[k] = [X_resampled, y_resampled]

    #save feature importance
    if k == 0:
        FI_FULL = clf.feature_importances_
    else:
        FI_FULL = np.column_stack((FI_FULL, clf.feature_importances_))

# Save model: 
joblib.dump(models, "PreGame_testing_models.pkl")

# Save feature importance: 
df_FI = np.mean(FI_FULL, axis=1)
df_FI = pd.DataFrame(df_FI)
df_FI.columns = ["FeatureImportance"]
df_FI.to_csv("PreGame_testing_feature_importance.csv", index=False)